# Asset Class Trend Following 策略回測 (equityNEW2)

本策略針對 **2019-2025** 全期間進行回測，採用經過多階段優化後選定的穩健參數組合。此參數組合旨在達成樣本內外一致的高 Calmar Ratio (>3) 並嚴格控制最大回撤 (<25%)。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def clean_data(filepath):
    df_prices = pd.read_excel(filepath, sheet_name='還原收盤價', header=None)
    df_volume = pd.read_excel(filepath, sheet_name='成交量', header=None)
    stock_codes = df_prices.iloc[0, 1:].values
    date_strings = df_prices.iloc[2:, 0].astype(str).str[:8]
    dates = pd.to_datetime(date_strings, format='%Y%m%d')
    prices = df_prices.iloc[2:, 1:].astype(float)
    prices.index = dates
    prices.columns = stock_codes
    volumes = df_volume.iloc[2:, 1:].astype(float)
    volumes.index = dates
    volumes.columns = stock_codes
    return prices.ffill().bfill(), volumes.fillna(0), dict(zip(stock_codes, df_prices.iloc[1, 1:].values))

prices, volumes, code_to_name = clean_data('樣本集-1.xlsx')


In [ ]:
class BacktesterV2:
    def __init__(self, prices, volumes, code_to_name):
        self.prices_df = prices
        self.volumes_df = volumes
        self.prices = prices.values
        self.volumes = volumes.values
        self.dates = prices.index
        self.assets = prices.columns
        self.code_to_name = code_to_name

    def run(self, sma_period, roc_period, stop_loss_pct, rebalance_interval):
        sma = self.prices_df.rolling(window=sma_period).mean().values
        roc = self.prices_df.pct_change(periods=roc_period).values
        sma5 = self.prices_df.rolling(window=5).mean().values
        sma10 = self.prices_df.rolling(window=10).mean().values
        sma20 = self.prices_df.rolling(window=20).mean().values
        
        surplus_pool = 30000000.0
        slots = {0: None, 1: None, 2: None}
        equity_curve = []
        start_idx = max(sma_period, roc_period, 20)

        for i in range(start_idx, len(self.dates)):
            date = self.dates[i]
            stock_mv = sum([s['shares'] * self.prices[i][s['asset_idx']] for s in slots.values() if s])
            total_equity = surplus_pool + stock_mv
            peak = max([e['equity'] for e in equity_curve] + [total_equity])
            equity_curve.append({'date': date, 'equity': total_equity, 'dd': (total_equity-peak)/peak})

            if i == len(self.dates) - 1: break
            
            # 停損檢查
            for sid, info in slots.items():
                if info:
                    if self.prices[i][info['asset_idx']] < info['max_p'] * (1 - stop_loss_pct):
                        surplus_pool += info['shares'] * self.prices[i+1][info['asset_idx']] * 0.995575
                        slots[sid] = None
                    else: info['max_p'] = max(info['max_p'], self.prices[i][info['asset_idx']])

            # 再平衡
            if (i - start_idx) % rebalance_interval == 0:
                valid = []
                idx_sorted = np.argsort(roc[i])[::-1]
                for idx in idx_sorted:
                    if len(valid) >= 3: break
                    p = self.prices[i][idx]
                    amt = p * self.volumes[i][idx] * 1000
                    if p > sma[i][idx] and roc[i][idx] > 0 and amt > 30000000 and p > sma5[i][idx] and p > sma10[i][idx] and p > sma20[i][idx]:
                        valid.append(idx)
                
                for sid, info in list(slots.items()):
                    if info and info['asset_idx'] not in valid:
                        surplus_pool += info['shares'] * self.prices[i+1][info['asset_idx']] * 0.995575
                        slots[sid] = None
                
                for a_idx in valid:
                    if a_idx not in [s['asset_idx'] for s in slots.values() if s]:
                        for sid in slots:
                            if not slots[sid]:
                                buy_p = self.prices[i+1][a_idx]
                                shares = (int(10000000 // (buy_p * 1.001425)) // 1000) * 1000
                                if shares > 0:
                                    surplus_pool -= shares * buy_p * 1.001425
                                    slots[sid] = {'asset_idx': a_idx, 'shares': shares, 'max_p': buy_p}
                                break
        return pd.DataFrame(equity_curve)


In [ ]:
# 使用全期間最穩健參數進行回測
sma, roc, sl, reb = 303, 10, 0.0999, 9
bt = BacktesterV2(prices, volumes, code_to_name)
res = bt.run(sma, roc, sl, reb)

# 設定回測期間
mask = (res['date'] >= '2019-01-01') & (res['date'] <= '2025-12-31')
res_p = res[mask]

# 繪製圖表
plt.figure(figsize=(12, 6))
plt.plot(res_p['date'], res_p['equity'])
plt.title('Equity Curve (2019-2025) - NEW2')
plt.grid(True)
plt.show()

# 計算並輸出績效
total_ret = (res_p['equity'].iloc[-1] / res_p['equity'].iloc[0]) - 1
years = (res_p['date'].iloc[-1] - res_p['date'].iloc[0]).days / 365.25
cagr = (1 + total_ret)**(1/years) - 1
mdd = res_p['dd'].min()
print(f"年化報酬率 (CAGR): {cagr:.2%}")
print(f"最大回撤 (MaxDD): {mdd:.2%}")
print(f"Calmar Ratio: {cagr/abs(mdd):.2f}")
